# Diagnóstico Bronze — steamcharts_2025_history_data

Tabla analizada: `steam_game_intelligence.bronze.steamcharts_2025_history_data`. Se revisan cobertura temporal, parsing del campo `month` (string), duplicados mensuales y métricas de jugadores. No se transforma todavía el campo temporal.

> Ejecuta las celdas en orden. Estas consultas son de diagnóstico: no modifican las tablas Bronze.

## 1. Volumen y cobertura temporal

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT steam_appid) AS distinct_app_ids,
  COUNT(DISTINCT month) AS distinct_month_strings,
  MIN(month) AS min_month_string,
  MAX(month) AS max_month_string,
  MIN(_ingested_at) AS first_ingested_at,
  MAX(_ingested_at) AS last_ingested_at
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data;

## 2. Nulos por columna

In [0]:
%sql
SELECT column_name, null_count
FROM (
  SELECT
    COUNT_IF(month IS NULL) AS month,
    COUNT_IF(avg_players IS NULL) AS avg_players,
    COUNT_IF(gain IS NULL) AS gain,
    COUNT_IF(gain_percent IS NULL) AS gain_percent,
    COUNT_IF(peak_players IS NULL) AS peak_players,
    COUNT_IF(name IS NULL) AS name,
    COUNT_IF(steam_appid IS NULL) AS steam_appid,
    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
    COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
) UNPIVOT (null_count FOR column_name IN (
  month, avg_players, gain, gain_percent, peak_players, name,
  steam_appid, _ingested_at, _source
))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Formato de month
Se prueba el formato `yyyy-MM`. Si el dataset usa otro formato, los resultados nulos indicarán que hay que adaptar el parser antes de limpiar.

In [0]:
%sql
SELECT month,
       TRY_TO_DATE(month, 'yyyy-MM') AS parsed_month,
       COUNT(*) AS row_count
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
GROUP BY month, TRY_TO_DATE(month, 'yyyy-MM')
ORDER BY parsed_month NULLS FIRST, month
LIMIT 200;

In [0]:
%sql
SELECT
  COUNT_IF(month IS NOT NULL AND TRY_TO_DATE(month, 'yyyy-MM') IS NULL) AS unparsed_month_rows,
  COUNT_IF(month IS NOT NULL AND TRIM(month) = '') AS blank_month_rows,
  COUNT_IF(steam_appid <= 0) AS invalid_appid_rows,
  COUNT_IF(avg_players < 0) AS negative_avg_players_rows,
  COUNT_IF(peak_players < 0) AS negative_peak_players_rows,
  COUNT_IF(gain_percent < -100) AS gain_percent_below_minus_100_rows,
  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data;

## 4. Duplicados por juego y mes

In [0]:
%sql
SELECT steam_appid, month, COUNT(*) AS row_count,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT avg_players) AS distinct_avg_players,
       COUNT(DISTINCT peak_players) AS distinct_peak_players
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
GROUP BY steam_appid, month
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 200;

## 5. Identificadores o nombres inconsistentes

In [0]:
%sql
SELECT steam_appid,
       COUNT(DISTINCT name) AS distinct_names,
       MIN(name) AS example_name,
       COUNT(*) AS rows_count
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
WHERE steam_appid IS NOT NULL
GROUP BY steam_appid
HAVING COUNT(DISTINCT name) > 1
ORDER BY rows_count DESC
LIMIT 100;

## 6. Métricas fuera de rango o sospechosas

In [0]:
%sql
SELECT *
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
WHERE steam_appid <= 0
   OR (avg_players < 0)
   OR (peak_players < 0)
   OR (avg_players IS NOT NULL AND peak_players IS NOT NULL AND avg_players > peak_players)
   OR (gain_percent < -100)
   OR (month IS NOT NULL AND TRY_TO_DATE(month, 'yyyy-MM') IS NULL)
   OR name IS NULL OR TRIM(name) = ''
LIMIT 200;

In [0]:
%python
from datetime import datetime
import os

table_name = 'steam_game_intelligence.bronze.steamcharts_2025_history_data'
notebook_name = '03_diagnostico_steamcharts_history'
queries = [('Consulta de celda SQL 1', 'SELECT\n  COUNT(*) AS total_rows,\n  COUNT(DISTINCT steam_appid) AS distinct_app_ids,\n  COUNT(DISTINCT month) AS distinct_month_strings,\n  MIN(month) AS min_month_string,\n  MAX(month) AS max_month_string,\n  MIN(_ingested_at) AS first_ingested_at,\n  MAX(_ingested_at) AS last_ingested_at\nFROM steam_game_intelligence.bronze.steamcharts_2025_history_data;'), ('Consulta de celda SQL 2', 'SELECT column_name, null_count\nFROM (\n  SELECT\n    COUNT_IF(month IS NULL) AS month,\n    COUNT_IF(avg_players IS NULL) AS avg_players,\n    COUNT_IF(gain IS NULL) AS gain,\n    COUNT_IF(gain_percent IS NULL) AS gain_percent,\n    COUNT_IF(peak_players IS NULL) AS peak_players,\n    COUNT_IF(name IS NULL) AS name,\n    COUNT_IF(steam_appid IS NULL) AS steam_appid,\n    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,\n    COUNT_IF(_source IS NULL) AS _source\n  FROM steam_game_intelligence.bronze.steamcharts_2025_history_data\n) UNPIVOT (null_count FOR column_name IN (\n  month, avg_players, gain, gain_percent, peak_players, name,\n  steam_appid, _ingested_at, _source\n))\nWHERE null_count > 0\nORDER BY null_count DESC;'), ('Consulta de celda SQL 3', "SELECT month,\n       TRY_TO_DATE(month, 'yyyy-MM') AS parsed_month,\n       COUNT(*) AS row_count\nFROM steam_game_intelligence.bronze.steamcharts_2025_history_data\nGROUP BY month, TRY_TO_DATE(month, 'yyyy-MM')\nORDER BY parsed_month NULLS FIRST, month\nLIMIT 200;"), ('Consulta de celda SQL 4', "SELECT\n  COUNT_IF(month IS NOT NULL AND TRY_TO_DATE(month, 'yyyy-MM') IS NULL) AS unparsed_month_rows,\n  COUNT_IF(month IS NOT NULL AND TRIM(month) = '') AS blank_month_rows,\n  COUNT_IF(steam_appid <= 0) AS invalid_appid_rows,\n  COUNT_IF(avg_players < 0) AS negative_avg_players_rows,\n  COUNT_IF(peak_players < 0) AS negative_peak_players_rows,\n  COUNT_IF(gain_percent < -100) AS gain_percent_below_minus_100_rows,\n  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp\nFROM steam_game_intelligence.bronze.steamcharts_2025_history_data;"), ('Consulta de celda SQL 5', 'SELECT steam_appid, month, COUNT(*) AS row_count,\n       COUNT(DISTINCT name) AS distinct_names,\n       COUNT(DISTINCT avg_players) AS distinct_avg_players,\n       COUNT(DISTINCT peak_players) AS distinct_peak_players\nFROM steam_game_intelligence.bronze.steamcharts_2025_history_data\nGROUP BY steam_appid, month\nHAVING COUNT(*) > 1\nORDER BY row_count DESC\nLIMIT 200;'), ('Consulta de celda SQL 6', 'SELECT steam_appid,\n       COUNT(DISTINCT name) AS distinct_names,\n       MIN(name) AS example_name,\n       COUNT(*) AS rows_count\nFROM steam_game_intelligence.bronze.steamcharts_2025_history_data\nWHERE steam_appid IS NOT NULL\nGROUP BY steam_appid\nHAVING COUNT(DISTINCT name) > 1\nORDER BY rows_count DESC\nLIMIT 100;'), ('Consulta de celda SQL 7', "SELECT *\nFROM steam_game_intelligence.bronze.steamcharts_2025_history_data\nWHERE steam_appid <= 0\n   OR (avg_players < 0)\n   OR (peak_players < 0)\n   OR (avg_players IS NOT NULL AND peak_players IS NOT NULL AND avg_players > peak_players)\n   OR (gain_percent < -100)\n   OR (month IS NOT NULL AND TRY_TO_DATE(month, 'yyyy-MM') IS NULL)\n   OR name IS NULL OR TRIM(name) = ''\nLIMIT 200;")]
queries.append(("Muestra de estructura (una fila)", f"SELECT * FROM {table_name} LIMIT 1"))

def markdown_value(value):
    if value is None:
        return "NULL"
    return str(value).replace("|", "\\|").replace("\r", " ").replace("\n", " ")

report = [
    f"# Resultados del diagn?stico ? {notebook_name}",
    "",
    f"- **Notebook:** `{notebook_name}`",
    f"- **Ejecutado:** `{datetime.now().astimezone().isoformat()}` (zona horaria del driver)",
    "- **Estado:** `completed`",
    "- **Objetivo:** conservar los resultados efectivos de las validaciones SQL del notebook para su revisi?n local.",
    "",
    "## Tablas inspeccionadas",
    f"- `{table_name}`",
    "",
    "## Consultas y resultados",
]
for title, query in queries:
    rows = [row.asDict(recursive=True) for row in spark.sql(query).limit(200).collect()]
    report.extend(["", f"### {title}", f"_Filas exportadas: {len(rows)}; m?ximo aplicado: 200._", ""])
    if not rows:
        report.append("No se detectaron registros que incumplan esta validaci?n o la consulta no devolvi? filas.")
        continue
    columns = list(rows[0].keys())
    report.append("| " + " | ".join(columns) + " |")
    report.append("| " + " | ".join(["---"] * len(columns)) + " |")
    for row in rows:
        report.append("| " + " | ".join(markdown_value(row.get(column)) for column in columns) + " |")
report.extend(["", "## Hallazgos y conclusiones", "Los hallazgos deben interpretarse a partir de los conteos y muestras exportados; esta celda no inventa conclusiones ni marca como aprobada una validaci?n no ejecutada.", ""])

context_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
marker = "/notebooks/data_validations"
if marker not in context_path:
    raise RuntimeError(f"No se pudo determinar la ra?z del repositorio desde la ruta del notebook: {context_path}")
repo_path = context_path.split(marker, 1)[0]
workspace_repo_path = repo_path if repo_path.startswith("/Workspace") else "/Workspace" + repo_path
output_dir = os.path.join(workspace_repo_path, "notebooks", "data_validations", "result_outputs")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"resultados_{notebook_name}.md")
temp_path = output_path + ".tmp"
with open(temp_path, "w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(report))
os.replace(temp_path, output_path)
print(f"Markdown generado: {output_path}; consultas exportadas: {len(queries)}")
